In [3]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import cohen_kappa_score
!pip install lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier



# Install CatBoost
!pip install catboost

from catboost import CatBoostClassifier


from google.colab import files
uploaded = files.upload()

Saving bank-additional-full.csv to bank-additional-full.csv


In [4]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [6]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


In [7]:
# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [8]:


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})
# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

In [9]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform

# Split
X_train, X_test, y_train, y_test = train_test_split(X_shuffled, y_shuffled, test_size=0.2, random_state=42)


In [10]:


# Model
model = LogisticRegression(max_iter=1000)

# Search space
param_dist = {
    "C": uniform(0.01, 10),
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [500, 1000, 2000]
}

In [11]:

# RandomizedSearchCV

random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=50,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'C': np.float64(4.947955963643907), 'max_iter': 500, 'solver': 'lbfgs'}
Best CV accuracy: 0.9015
Test accuracy: 0.8966


In [12]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Logistic Regression)
from sklearn.linear_model import LogisticRegression

clf_name = "Logistic Regression"
clf = LogisticRegression(random_state=42, max_iter=1000)



# Store results
results = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results_df = pd.DataFrame(results)

# Print
print(f"Results for {clf_name}:")
print(results_df)

# Save to CSV if needed
results_df.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Logistic Regression:
   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Logistic Regression  0.898762   0.633136  0.231602     0.983046   
1     2  Logistic Regression  0.894392   0.635714  0.188161     0.986012   
2     3  Logistic Regression  0.905560   0.707006  0.244493     0.987449   
3     4  Logistic Regression  0.898034   0.632530  0.226293     0.983311   
4     5  Logistic Regression  0.899490   0.627219  0.231947     0.982796   
5     6  Logistic Regression  0.893178   0.700000  0.222222     0.986722   
6     7  Logistic Regression  0.905074   0.634483  0.213953     0.985633   
7     8  Logistic Regression  0.906045   0.713287  0.227679     0.988831   
8     9  Logistic Regression  0.899951   0.680556  0.211207     0.987411   
9    10  Logistic Regression  0.898009   0.713333  0.221074     0.988167   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.339144  0.477153  0.016954  0.607324  0.341482  0.296901   


In [13]:


# Model
from sklearn.tree import DecisionTreeClassifier
model1 = DecisionTreeClassifier()

# Search space
param_dist1 = {

    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

In [14]:

# DecisionTreeClassifier

random_search = RandomizedSearchCV(model1, param_distributions=param_dist1, n_iter=50,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 5}
Best CV accuracy: 0.5640
Test accuracy: 0.8987


In [15]:


from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Decision Tree)
from sklearn.tree import DecisionTreeClassifier
clf_name = "Decision Tree"
clf = DecisionTreeClassifier()



# Store results
results1 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results1.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results1_df1 = pd.DataFrame(results1)

# Print
print(f"Results for {clf_name}:")
print(results1_df1)

# Save to CSV if needed
results1_df1.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Decision Tree:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Decision Tree  0.838553   0.306667  0.348485     0.900465  0.326241   
1     2  Decision Tree  0.837339   0.295218  0.300211     0.907021  0.297694   
2     3  Decision Tree  0.851420   0.329741  0.337004     0.915143  0.333333   
3     4  Decision Tree  0.831270   0.278311  0.312500     0.897127  0.294416   
4     5  Decision Tree  0.839767   0.313076  0.371991     0.898143  0.340000   
5     6  Decision Tree  0.844865   0.365805  0.365079     0.911757  0.365442   
6     7  Decision Tree  0.849721   0.299363  0.327907     0.910545  0.312986   
7     8  Decision Tree  0.844137   0.305221  0.339286     0.905748  0.321353   
8     9  Decision Tree  0.840214   0.307540  0.334052     0.904488  0.320248   
9    10  Decision Tree  0.838514   0.328273  0.357438     0.902587  0.342235   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.560177  0.099

In [16]:


# Model
from sklearn.ensemble import RandomForestClassifier
model2 = RandomForestClassifier()

# Search space
param_dist2 = {
         "n_estimators":[100,200,300],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}

In [17]:

# RandomForestClassifier

random_search = RandomizedSearchCV(model2, param_distributions=param_dist2, n_iter=15,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 10}
Best CV accuracy: 0.6658
Test accuracy: 0.8987


In [18]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Random Forest)
from sklearn.ensemble import RandomForestClassifier
clf_name = "Random Forest"
clf = RandomForestClassifier()



# Store results
results2 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results2.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results2_df2 = pd.DataFrame(results2)

# Print
print(f"Results for {clf_name}:")
print(results2_df2)

# Save to CSV if needed
results2_df2.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Random Forest:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.892692   0.536496  0.318182     0.965272  0.399457   
1     2  Random Forest  0.888080   0.524590  0.270613     0.968184  0.357043   
2     3  Random Forest  0.895120   0.547414  0.279736     0.971351  0.370262   
3     4  Random Forest  0.888322   0.507463  0.293103     0.963885  0.371585   
4     5  Random Forest  0.887837   0.490421  0.280088     0.963681  0.356546   
5     6  Random Forest  0.886137   0.564576  0.303571     0.967358  0.394839   
6     7  Random Forest  0.897305   0.516279  0.258140     0.971808  0.344186   
7     8  Random Forest  0.899005   0.561538  0.325893     0.968946  0.412429   
8     9  Random Forest  0.891209   0.535088  0.262931     0.970991  0.352601   
9    10  Random Forest  0.892909   0.574394  0.342975     0.966153  0.429495   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.554195  0.034

In [19]:


# Model
from sklearn.neighbors import KNeighborsClassifier
model3 = KNeighborsClassifier()

# Search space
param_dist3 = {
         "n_neighbors": [3, 5, 7],
            "weights": ["uniform", "distance"]
}

In [20]:

# KNeighbors Classifier

random_search = RandomizedSearchCV(model3, param_distributions=param_dist3, n_iter=25,
                                   cv=15, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'weights': 'uniform', 'n_neighbors': 7}
Best CV accuracy: 0.8120
Test accuracy: 0.8919


In [21]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (KNN)
from sklearn.neighbors import KNeighborsClassifier
clf_name = "KNN"
clf = KNeighborsClassifier()

# Store results
results3 = []




# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results3.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results3_df3 = pd.DataFrame(results3)

# Print
print(f"Results for {clf_name}:")
print(results3_df3)

# Save to CSV if needed
results3_df3.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for KNN:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1        KNN  0.891479   0.564706  0.300000     0.969497  0.391837   
1      2        KNN  0.887473   0.508772  0.278846     0.965489  0.360248   
2      3        KNN  0.887473   0.482558  0.273927     0.963569  0.349474   
3      4        KNN  0.900947   0.608187  0.336570     0.972507  0.433333   
4      5        KNN  0.888929   0.506944  0.237785     0.970890  0.323725   
5      6        KNN  0.893299   0.525140  0.311258     0.965221  0.390852   
6      7        KNN  0.887837   0.448864  0.272414     0.960505  0.339056   
7      8        KNN  0.879097   0.510753  0.282738     0.962241  0.363985   
8      9        KNN  0.886744   0.569767  0.292537     0.969307  0.386588   
9     10        KNN  0.887473   0.484848  0.263158     0.965192  0.341151   
10    11        KNN  0.902403   0.543046  0.291815     0.972008  0.379630   
11    12        KNN  0.894028   0.505208  0.331058     0.96

In [22]:


# Model
from sklearn.svm import LinearSVC
model4 = LinearSVC()



from sklearn.svm import LinearSVC

model4 = LinearSVC()

param_dist4 = {
    "C": [0.01, 0.1, 1, 10],
    "loss": ["hinge", "squared_hinge"],
    "max_iter": [1000, 2000]
}

In [23]:

# LinearSVC

random_search = RandomizedSearchCV(model4, param_distributions=param_dist4, n_iter=10,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'max_iter': 1000, 'loss': 'hinge', 'C': 1}
Best CV accuracy: 0.8813
Test accuracy: 0.1140


In [24]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (LinearSVC)
from sklearn.svm import LinearSVC
clf_name = "SVM"
clf = LinearSVC()


# Store results
results4 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results4.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results4_df4 = pd.DataFrame(results4)

# Print
print(f"Results for {clf_name}:")
print(results4_df4)

# Save to CSV if needed
results4_df4.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for SVM:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1        SVM  0.897669   0.714286  0.203125     0.989283  0.316302   
1      2        SVM  0.895849   0.647727  0.182692     0.987264  0.285000   
2      3        SVM  0.897669   0.627907  0.178218     0.986901  0.277635   
3      4        SVM  0.904588   0.747368  0.229773     0.990152  0.351485   
4      5        SVM  0.899490   0.691358  0.182410     0.989750  0.288660   
5      6        SVM  0.900583   0.646465  0.211921     0.985679  0.319202   
6      7        SVM  0.901311   0.602151  0.193103     0.984935  0.292428   
7      8        SVM  0.891843   0.714286  0.193452     0.989212  0.304450   
8      9        SVM  0.894028   0.720000  0.214925     0.988387  0.331034   
9     10        SVM  0.900947   0.670213  0.207237     0.987305  0.316583   
10    11        SVM  0.908230   0.723077  0.167260     0.992698  0.271676   
11    12        SVM  0.907502   0.729412  0.211604     0.99

In [25]:


# Model

from xgboost import XGBClassifier

model5 =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')

param_dist5 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
            "max_depth": [3, 6]
}

In [26]:

# XGBoost

random_search = RandomizedSearchCV(model5, param_distributions=param_dist5, n_iter=100,
                                   cv=20, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01}
Best CV accuracy: 0.8065
Test accuracy: 0.8944


In [27]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from xgboost import XGBClassifier
clf_name = "XGBoost"
clf =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')


# Store results
results5 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results5.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results5_df5 = pd.DataFrame(results5)

# Print
print(f"Results for {clf_name}:")
print(results5_df5)

# Save to CSV if needed
results5_df5.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for XGBoost:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1    XGBoost  0.890386   0.559748  0.278125     0.971146  0.371608   
1      2    XGBoost  0.891843   0.551020  0.259615     0.972884  0.352941   
2      3    XGBoost  0.896941   0.571429  0.264026     0.975440  0.361174   
3      4    XGBoost  0.904953   0.671429  0.304207     0.981124  0.418708   
4      5    XGBoost  0.901311   0.625000  0.293160     0.977860  0.399113   
5      6    XGBoost  0.898034   0.568750  0.301325     0.971768  0.393939   
6      7    XGBoost  0.895484   0.509804  0.268966     0.969463  0.352144   
7      8    XGBoost  0.889658   0.610738  0.270833     0.975934  0.375258   
8      9    XGBoost  0.886016   0.572368  0.259701     0.973040  0.357290   
9     10    XGBoost  0.898034   0.593750  0.250000     0.978706  0.351852   
10    11    XGBoost  0.906409   0.592308  0.274021     0.978499  0.374696   
11    12    XGBoost  0.907138   0.631944  0.310580     

In [28]:


# Model LightGBM



model6 =  LGBMClassifier()

param_dist6 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1]
}

In [29]:

# LightGBM

random_search = RandomizedSearchCV(model6, param_distributions=param_dist6, n_iter=100,
                                   cv=25, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

[LightGBM] [Info] Number of positive: 4327, number of negative: 34116
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005038 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 492
[LightGBM] [Info] Number of data points in the train set: 38443, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.112556 -> initscore=-2.064892
[LightGBM] [Info] Start training from score -2.064892
Best parameters: {'n_estimators': 100, 'learning_rate': 0.01}
Best CV accuracy: 0.6616
Test accuracy: 0.8947


In [30]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from lightgbm import LGBMClassifier
clf_name = "LightGBM"
clf =  LGBMClassifier()


# Store results
results6 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results6.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results6_df6 = pd.DataFrame(results6)

# Print
print(f"Results for {clf_name}:")
print(results6_df6)

# Save to CSV if needed
results6_df6.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

[LightGBM] [Info] Number of positive: 4320, number of negative: 34122
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004838 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 493
[LightGBM] [Info] Number of data points in the train set: 38442, number of used features: 44
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.112377 -> initscore=-2.066687
[LightGBM] [Info] Start training from score -2.066687
[LightGBM] [Info] Number of positive: 4328, number of negative: 34114
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004995 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 493
[LightGBM] [Info] Number of data points in the train set: 38442, number of used features: 44
[LightGBM] [Info] [bin

In [31]:


# Model CatBoost

from catboost import CatBoostClassifier

model7 =   CatBoostClassifier(verbose=0)

param_dist7 = {
     "iterations": [100, 200],
     "depth": [4, 6],
     "learning_rate": [0.01, 0.1]
}

In [32]:

# LightGBM

random_search = RandomizedSearchCV(model7, param_distributions=param_dist7, n_iter=100,
                                   cv=30, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'learning_rate': 0.01, 'iterations': 100, 'depth': 4}
Best CV accuracy: 0.8467
Test accuracy: 0.8980


In [33]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (CatBoost)
from catboost import CatBoostClassifier
clf_name = "CatBoost"
clf =   CatBoostClassifier(iterations=20, verbose=0)


# Store results
results7 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results7.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results7_df7 = pd.DataFrame(results7)

# Print
print(f"Results for {clf_name}:")
print(results7_df7)

# Save to CSV if needed
results7_df7.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for CatBoost:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1   CatBoost  0.896941   0.627586  0.284375     0.977741  0.391398   
1      2   CatBoost  0.895849   0.614035  0.224359     0.981923  0.328638   
2      3   CatBoost  0.899490   0.613445  0.240924     0.981171  0.345972   
3      4   CatBoost  0.907866   0.712121  0.304207     0.984407  0.426304   
4      5   CatBoost  0.900218   0.643478  0.241042     0.983190  0.350711   
5      6   CatBoost  0.901675   0.617647  0.278146     0.978723  0.383562   
6      7   CatBoost  0.898762   0.543478  0.258621     0.974349  0.350467   
7      8   CatBoost  0.896941   0.719008  0.258929     0.985892  0.380744   
8      9   CatBoost  0.891479   0.627586  0.271642     0.977603  0.379167   
9     10   CatBoost  0.902039   0.644628  0.256579     0.982391  0.367059   
10    11   CatBoost  0.908594   0.638889  0.245552     0.984178  0.354756   
11    12   CatBoost  0.907502   0.653543  0.283276    

In [34]:


# Model Gradient Boosting

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier

model8 =   GradientBoostingClassifier(n_estimators=25)

param_dist8 = {
      "learning_rate": [0.1, 0.3,0.6],
      "n_estimators": [50, 100],
}

In [35]:

# Gradient Boosting

random_search = RandomizedSearchCV(model8, param_distributions=param_dist8, n_iter=200,
                                   cv=30, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 50, 'learning_rate': 0.1}
Best CV accuracy: 0.7769
Test accuracy: 0.8976


In [36]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Gradient Boosting)
from sklearn.ensemble import  GradientBoostingClassifier

clf_name = "Gradient Boosting"
clf =    GradientBoostingClassifier()


# Store results
results8 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results8.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results8_df8= pd.DataFrame(results8)

# Print
print(f"Results for {clf_name}:")
print(results8_df8)

# Save to CSV if needed
results8_df8.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Gradient Boosting:
    Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0      1  Gradient Boosting  0.896941   0.637037  0.268750     0.979802   
1      2  Gradient Boosting  0.894756   0.605505  0.211538     0.982334   
2      3  Gradient Boosting  0.900947   0.632479  0.244224     0.982399   
3      4  Gradient Boosting  0.904953   0.706897  0.265372     0.986048   
4      5  Gradient Boosting  0.904224   0.703704  0.247557     0.986880   
5      6  Gradient Boosting  0.903132   0.642857  0.268212     0.981588   
6      7  Gradient Boosting  0.903860   0.600000  0.268966     0.978827   
7      8  Gradient Boosting  0.894756   0.697479  0.247024     0.985062   
8      9  Gradient Boosting  0.895484   0.676471  0.274627     0.981750   
9     10  Gradient Boosting  0.904588   0.690909  0.250000     0.986077   
10    11  Gradient Boosting  0.908594   0.653061  0.227758     0.986207   
11    12  Gradient Boosting  0.909323   0.680328  0.283276     0.9841

In [37]:


# Model MLP

from sklearn.neural_network import MLPClassifier
model9 =   MLPClassifier(max_iter=300)
param_dist9 = {
      "hidden_layer_sizes": [(64,64), (128,64)]
}

In [38]:

# MLP

random_search = RandomizedSearchCV(model9, param_distributions=param_dist9, n_iter=300,
                                   cv=30, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'hidden_layer_sizes': (128, 64)}
Best CV accuracy: 0.8911
Test accuracy: 0.8965


In [39]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (MLP)
from sklearn.neural_network import MLPClassifier
clf_name = "MLP"
clf =   MLPClassifier(max_iter=500)


# Store results
results9 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=15, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results9.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results9_df9= pd.DataFrame(results9)

# Print
print(f"Results for {clf_name}:")
print(results9_df9)

# Save to CSV if needed
results9_df9.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for MLP:
    Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0      1        MLP  0.845594   0.391667  0.587500     0.879637  0.470000   
1      2        MLP  0.878004   0.455253  0.375000     0.942482  0.411248   
2      3        MLP  0.895120   0.564103  0.217822     0.979124  0.314286   
3      4        MLP  0.896213   0.722222  0.126214     0.993845  0.214876   
4      5        MLP  0.895120   0.614458  0.166124     0.986880  0.261538   
5      6        MLP  0.899490   0.618182  0.225166     0.982815  0.330097   
6      7        MLP  0.135470   0.107681  0.986207     0.035016  0.194162   
7      8        MLP  0.877640   0.000000  0.000000     1.000000  0.000000   
8      9        MLP  0.878004   0.000000  0.000000     1.000000  0.000000   
9     10        MLP  0.889294   0.000000  0.000000     1.000000  0.000000   
10    11        MLP  0.897669   0.000000  0.000000     1.000000  0.000000   
11    12        MLP  0.893299   0.500000  0.409556     0.95

In [40]:


# Model Bagging
from sklearn.ensemble import BaggingClassifier
model10 =   BaggingClassifier(n_estimators=10)
param_dist10 = {
     "n_estimators": [50, 100]
}

In [41]:

# BaggingClassifier

random_search = RandomizedSearchCV(model10, param_distributions=param_dist10, n_iter=500,
                                   cv=30, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100}
Best CV accuracy: 0.4746
Test accuracy: 0.8882


In [42]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (BaggingClassifier)
from sklearn.ensemble import BaggingClassifier
clf_name = "BaggingClassifier"
clf =   BaggingClassifier(n_estimators=10)


# Store results
results10 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=25, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results10.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results10_df10= pd.DataFrame(results10)

# Print
print(f"Results for {clf_name}:")
print(results10_df10)

# Save to CSV if needed
results10_df10.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for BaggingClassifier:
    Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0      1  BaggingClassifier  0.883495   0.491667  0.310526     0.958162   
1      2  BaggingClassifier  0.888956   0.515873  0.347594     0.958248   
2      3  BaggingClassifier  0.891383   0.485437  0.284091     0.963995   
3      4  BaggingClassifier  0.878641   0.465909  0.211340     0.967675   
4      5  BaggingClassifier  0.878641   0.450820  0.292553     0.954110   
5      6  BaggingClassifier  0.891990   0.524752  0.289617     0.967235   
6      7  BaggingClassifier  0.895024   0.532110  0.322222     0.965259   
7      8  BaggingClassifier  0.881068   0.483146  0.222798     0.968385   
8      9  BaggingClassifier  0.888350   0.463636  0.289773     0.959918   
9     10  BaggingClassifier  0.881068   0.453704  0.263441     0.959644   
10    11  BaggingClassifier  0.890777   0.495495  0.307263     0.961879   
11    12  BaggingClassifier  0.881675   0.405660  0.245714     0.9572

In [43]:
# Combine all results into ONE Excel file (multiple sheets)

with pd.ExcelWriter("Final_Summary.xlsx") as writer:

    results_df.to_excel(writer, sheet_name="Logistic Regression", index=False)
    results1_df1.to_excel(writer, sheet_name="Decision Tree", index=False)
    results2_df2.to_excel(writer, sheet_name="Random Forest", index=False)
    results3_df3.to_excel(writer, sheet_name="KNN", index=False)
    results4_df4.to_excel(writer, sheet_name="SVM", index=False)
    results5_df5.to_excel(writer, sheet_name="XGBoost", index=False)
    results6_df6.to_excel(writer, sheet_name="LightGBM", index=False)
    results7_df7.to_excel(writer, sheet_name="CatBoost", index=False)
    results8_df8.to_excel(writer, sheet_name="Gradient Boosting", index=False)
    results9_df9.to_excel(writer, sheet_name="MLP", index=False)
    results10_df10.to_excel(writer, sheet_name="Bagging", index=False)


print("Excel file created successfully!")

Excel file created successfully!


In [44]:
from google.colab import files
files.download("Final_Summary.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>